Used for reference: https://github.com/sara-kassani/Python-Machine-Learning-Codes/blob/master/Keras%20NN%20-%20Breast%20Cancer%20Wisconsin%20(Diagnostic)%20Data%20Set.ipynb
and https://github.com/Eakta08/Breast-Cancer-Prediction/blob/main/Breast%20Cancer%20Prediction.ipynb

In [1]:
import numpy as np
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from sklearn.datasets import load_breast_cancer


2026-03-19 17:04:45.182382: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-19 17:04:45.182576: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-19 17:04:45.210691: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-19 17:04:45.824813: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Load Dataset

In [2]:
data = load_breast_cancer() #The model has 30 features (https://www.kaggle.com/datasets/ucimachinelearning/wisconsin-breast-cancer-dataset
X = data.data
Y = data.target.astype(np.float32)   #0 = malignant, 1 = benign

Train/Test split

In [3]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size = 0.2,
    random_state = 42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape)
inputs = X_train.shape[1]
print(inputs)

(455, 30)
30


In [4]:
#Copied from 28x28 MNIST notbeook
@keras.saving.register_keras_serializable()
def fake_q07(x):
    x_clip = tf.clip_by_value(x, -1.0, 127.0/128.0)
    x_q = tf.round(x_clip * 128.0) / 128.0
    return x_clip + tf.stop_gradient(x_q - x_clip)

model = keras.Sequential([
    layers.Input(shape = (inputs,)),
    layers.Dense(64),
    layers.ReLU(max_value = 127.0/128.0),
    layers.Lambda(fake_q07),
    layers.Dense(32),
    layers.ReLU(max_value = 127.0/128.0),
    layers.Lambda(fake_q07),
    layers.Dense(16),
    layers.ReLU(max_value = 127.0/128.0),
    layers.Lambda(fake_q07),
    layers.Dense(1),
    layers.Activation(activation="sigmoid") #No layer like ReLU
])

model.summary()

E0000 00:00:1773920086.511136   16728 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1773920086.515916   16728 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,609 (18.00 KB)

 Trainable params: 4,609 (18.00 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", 
        patience=5,              
        min_delta=0.001,          
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", 
        factor=0.5, 
        patience=2               
    ),
]

model.fit(X_train_scaled, Y_train, epochs=100, batch_size = 8, verbose=2, validation_split = 0.2, callbacks=callbacks)



Epoch 1/100
46/46 - 1s - 14ms/step - accuracy: 0.8764 - loss: 0.4659 - val_accuracy: 0.9451 - val_loss: 0.3246 - learning_rate: 0.0010
Epoch 2/100
46/46 - 0s - 2ms/step - accuracy: 0.9560 - loss: 0.2596 - val_accuracy: 0.9560 - val_loss: 0.2313 - learning_rate: 0.0010
Epoch 3/100
46/46 - 0s - 2ms/step - accuracy: 0.9808 - loss: 0.1869 - val_accuracy: 0.9560 - val_loss: 0.1912 - learning_rate: 0.0010
Epoch 4/100
46/46 - 0s - 2ms/step - accuracy: 0.9863 - loss: 0.1485 - val_accuracy: 0.9670 - val_loss: 0.1709 - learning_rate: 0.0010
Epoch 5/100
46/46 - 0s - 2ms/step - accuracy: 0.9890 - loss: 0.1248 - val_accuracy: 0.9670 - val_loss: 0.1542 - learning_rate: 0.0010
Epoch 6/100
46/46 - 0s - 2ms/step - accuracy: 0.9918 - loss: 0.1075 - val_accuracy: 0.9670 - val_loss: 0.1413 - learning_rate: 0.0010
Epoch 7/100
46/46 - 0s - 1ms/step - accuracy: 0.9918 - loss: 0.0933 - val_accuracy: 0.9670 - val_loss: 0.1322 - learning_rate: 0.0010
Epoch 8/100
46/46 - 0s - 1ms/step - accuracy: 0.9945 - loss: 

In [6]:
#Evaluate model
test_loss, test_acc = model.evaluate(X_test_scaled, Y_test)
print("Test accuracy: ", test_acc)

#print(model.get_weights())
model.save(filepath="breast_cancer.keras")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9825 - loss: 0.1036 
Test accuracy:  0.9824561476707458


Ada samples export

In [ ]:
from pathlib import Path
import random
ada_out_dir = Path("exported_samples_q07")
ada_out_dir.mkdir(parents=True, exist_ok=True)
ada_ads_path = ada_out_dir / "breast_cancer_test_samples.ads"
ada_package_name = "breast_cancer_test_samples"

In [8]:
def write_ada_int_array(f, name, flat):
    f.write(f"  {name} : constant Integer_Array (Natural range 0 .. {len(flat)-1}) := (\n")
    for i, w in enumerate(flat):
        sep = "," if i != len(flat) - 1 else ""
        f.write(f"  {int(w)}{sep}\n")
    f.write("  );\n")

def quantize_q07(x):
    x_clipped=np.clip(x, -1.0, 0.9921875) #Restrict weights in the numpy array to -1 to 127/128
    q=np.round(x_clipped * 128.0).astype(np.int16) #128=scaling factor
                                                     #int16 is used in the intermediate step 
                                                     #similar to our calculations in VHDL
    q=np.clip(q, -128, 127).astype(np.int8)       #Clip to int8
    return q

In [ ]:
#Convert tensors to numpy arrays for indexing
test_vec_np = np.array(X_test_scaled, dtype=np.float32)
test_labels_np = np.array(Y_test, dtype=np.int32)

sample_count = 10

random_numbers= [random.randint(0, len(test_vec_np) - 1) for _ in range(sample_count)]

random_samples = []
random_labels = []

for index in random_numbers:
    random_samples.append(quantize_q07(test_vec_np[index]))
    random_labels.append(int(test_labels_np[index]))

feature_count = test_vec_np.shape[1]   #should be 30

with open(ada_ads_path, "w") as f:
    f.write("with Input_Output_Helper; use Input_Output_Helper;\n")
    f.write(f"package {ada_package_name} is\n")
    f.write(f"  Sample_Count : constant Natural := {sample_count};\n")
    f.write(f"  type Sample_Set is array (Natural range 0 .. Sample_Count - 1) of Integer_Array (Natural range 0 .. 29);\n")

    #Write Samples as a proper 2D Ada constant
    f.write("   Samples : constant Sample_Set := (\n")
    for s in range(sample_count):
        f.write(f"  {s} => (\n")
        flat = random_samples[s].astype(np.int8).flatten()  #ensure Python int printing is sane
        for i, v in enumerate(flat):
            if(i!=len(flat)-1):
                sep = ","
            else:
                sep = ""
            f.write(f"  {int(v)}{sep}")
            if((i + 1) % 20 == 0):
                f.write("\n")
            else:
                f.write(" ")
        closer="\n)"
        if(s != sample_count - 1):
            closer = closer + ","
        closer = closer + "\n"
        f.write(closer)
    f.write(");\n")

    write_ada_int_array(f,"    Labels", random_labels)

    f.write(f"end {ada_package_name};\n")

print(f"Wrote Ada samples to: {ada_ads_path}")

Wrote Ada samples to: exported_samples_q07/breast_cancer_test_samples_28x28.ads
